# Lesson 02 - Microsoft Agent Framework ကို ရှာဖွေခြင်း

**Microsoft Agent Framework (MAF)** သည် AI ကိုယ်ပိုင်အေးဂျင့်များ တည်ဆောက်ရန် အစုလိုက်ပေါင်းလိုက်ဖောင်မြူလာတစ်ခုဖြစ်သည်။ ၎င်းသည် သန့်ရှင်းပြီး ပေါင်းစည်းနိုင်သော စနစ်တစ်ခုကို နိုင်ပြီး အဓိက အဆောက်အအုံ အပိုင်းလေးခု ပါဝင်သည်။

- **Client** – AI မော်ဒယ် endpoint နှင့် ချိတ်ဆက်ပြီး ဆက်သွယ်မှုကို ကိုင်တွယ်သည်
- **Agent** – ပေါ်လွင်သော client ကို ညွှန်ကြားချက်များနှင့် ကိရိယာသတ်မှတ်ချက်များဖြင့် ထုပ်ပိုးသည်
- **Tools** – မော်ဒယ်က ခေါ်ယူနိုင်သော သီးသန့်လုပ်ဆောင်ချက်များဖြင့် ကိုယ်ပိုင်အေးဂျင့်စွမ်းဆောင်ရည်ကို တိုးချဲ့သည်
- **Session** – မကြာခဏ စကားပြောဆွေးနွေးမှုများအတွက် စကားပြောမှတ်တမ်းကို ထိန်းသိမ်းသည်

ဒီသင်ခန်းစာတွင်၊ အဆိုပါအယူအဆများကို အသုံးပြု၍ **ခရီးသွားဘွတ်ခွာမှုအေးဂျင့်** တစ်ခုကို တည်ဆောက်မည်ဖြစ်ပြီး သွားမည့်နေရာရနိုင်မှုကို စစ်ဆေးပါမည်။


## စနစ်တပ်ဆင်ခြင်း


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Agent Framework ရုပ်ပုံကိုနားလည်ခြင်း

Microsoft Agent Framework သည် အလွှာစနစ်ဖြင့် ပုံသဏ္ဌာန်ဖော်ထားသည်။

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – `AzureAIProjectAgentProvider` သည် Azure OpenAI deployment မှတွဲဆက်ထားသည်။ authentication၊ request formatting နှင့် response parsing ကို ကိုင်တွယ်ပေးသည်။
2. **Agent** – client မှ `provider.create_agent()` ဖြင့် ဖန်တီးပြီး၊ model access ကို instruction (system prompt) နှင့် tools များနှင့် ပေါင်းစပ်ထားသည်။
3. **Tools** – `@tool` ဖြင့် အလှည့်ကွက်ထားသော Python function များဖြစ်ပြီး၊ agent သည် လုပ်ဆောင်ရန် သို့မဟုတ် data ရရန် ခေါ်ယူနိုင်သည်။
4. **Session** – `AgentSession` object (agent.create_session() ဖြင့် ဖန်တီး) သည် စကားပြောရာ မှတ်တမ်းများကို သိမ်းဆည်းသည်၊ မျိုးစုံပြောဆိုမှုများအတွက် agent သည် ယခင် context ကို မှတ်မိစေသည်။

တစ်ခုချင်း အလွှာများကို အဆင့်လိုက် တည်ဆောက်ကြစို့။


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool Decorator ဖြင့် ကိရိယာများထည့်ခြင်း

ကိရိယာများက စာသားထုတ်ပေးခြင်းမှ ပြန်လှည့်၍ ရှေ့ဆောင်များအား လုပ်ဆောင်ချက်များပြုလုပ်ခွင့်ပေးသည်။ `@tool` decorator သည် ပုံမှန် Python function ကို ရှေ့ဆောင်မှ ခေါ်ယူနိုင်သော အရာအဖြစ် ပြောင်းလဲပေးသည်။

အဓိကချက်များ-
- မော်ဒယ်အနေဖြင့် အခြား parameter တစ်ခုချင်းစီကို နားလည်စေရန် `Annotated[type, "description"]` ကို အသုံးပြုပါ။
- docstring သည် မော်ဒယ်မြင်ရသော ကိရိယာ၏ ဖော်ပြချက်ဖြစ်သည်။
- `approval_mode="never_require"` သည် အသုံးပြုသူ၏ အတည်ပြုမှု မလိုအပ်ဘဲ ကိရိယာကို အလိုအလျောက် ပြေးဆွဲစေသည်။


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## ကိရိယာများဖြင့် လုပ်ရပ်တစ်ခု ဖန်တီးခြင်း

အခုတော့ client, အညွှန်းများနဲ့ ကိရိယာများကို agent တစ်ခုအဖြစ် ပေါင်းစပ်ကြပါမယ်။ `instructions` က ဆော့ဖ်ဝဲစနစ် prompt အနေနဲ့ တာဝန်ထမ်းဆောင်ကြပြီး agent ရဲ့ပုဂ္ဂိုလ်ရေးနှင့် ကျင့်ဝတ်တွေကို သတ်မှတ်ပေးသည်။


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## ဆက်စပ် ပြောဆိုဆွေးနွေးမှုများနှင့် အစည်းအဝေးများ

`AgentSession` တစ်ခု ( `agent.create_session()` မှဖန်တီးထားသည်) သည် ဆွေးနွေးမှုအတွင်းရှိ စာတိုက်များအားလုံးကို မှတ်တမ်းတင်ထားသည်။ တူညီသော session ကို `agent.run()` ဖုန်းတိုက်တိုင်းတွင်ပါဝင်ရာ ဖော်ပြခဲ့လျှင်၊ agent သည် ပြောဆိုမှု အပြည့်အစုံ ရှိနောက်ခံသမိုင်းအား အသုံးပြု၍ ရှေးခေတ် စာတိုက်များကို ပြန်လည်ရုံးညှိနိုင်သည်။

ကျွန်ုပ်တို့သည် `tools=[check_destination_availability]` ကို ရွှေ့ပြောင်းပေးသည်။ ထို့ကြောင့် agent သည် တိုက်တစ်ခုချင်းစီတွင် ကျွန်ုပ်တို့၏ ရရှိနိုင်မှု စစ်ဆေးသူကို ခေါ်ယူနိုင်သည်။


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## အကျဥ်းချုပ်

ဤမိနစ်သင်ခန်းစာတွင် Microsoft Agent Framework ၏ ခြေလှမ်းလေးခုကို လေ့လာခဲ့သည်-

| အကြောင်းအရာ | သင်လေ့လာခဲ့သည် |
|---------|------------------|
| **Client** | `AzureAIProjectAgentProvider` သည် credentials အား အခြေခံ၍ Azure OpenAI နှင့် ချိတ်ဆက်ပေးသည် |
| **Agent** | `provider.create_agent()` သည် မော်ဒယ်ချိတ်ဆက်မှုကို အညွှန်းများနှင့် နာမည်ဖြင့် ပေါင်းစပ်ပေးသည် |
| **Tools** | `@tool` decorator သည် Python function များကို agent အတွက် ခေါ်နိုင်ရန် ဖော်ပြပေးသည် |
| **Session** | `agent.create_session()` သည် တစ်ခေါက်နောက်တစ်ခေါက် စကားပြောဆိုမှုသမိုင်းကို ထိန်းသိမ်းပေးသည် |

ဤအခြေခံအဆောက်အအုံများသည် သဘာဝစကားပြောဆိုနိုင်သော agent များကို ဖန်တီးရန်၊ ပြင်ပ function များကို ခေါ်ယူရန်နှင့် စာကြောင်းအဆက်အစပ်ကို ထိန်းသိမ်းရန် အခြေခံသည်- နောက်ပိုင်းသင်ခန်းစာများတွင် ပိုမိုတိုးတက်သည့် agentic အတိုင်းအတာများအတွက် အခြေခံ ဖြစ်သည်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**အဆိုပြုချက်**  
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှုဖြစ်သော [Co-op Translator](https://github.com/Azure/co-op-translator) မှတဆင့် ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှုအတွက် ကြိုးစားပါသော်လည်း၊ အလိုအလျောက် ဘာသာပြန်ခြင်းများတွင် အမှားယွင်းမှု သို့မဟုတ်တိကျမှုမရှိခြင်းများ ရှိနိုင်ကြောင်း ကျေးဇူးပြု၍ သဘောထားပေးပါရန် မေတ္တာရပ်ခံအပ်ပါသည်။ မူလစာတမ်းကို မူလဘာသာဖြင့်သာ ခံယူရမည့် အတည်ပြုသည့်အချက်အလက်အဖြစ် သတ်မှတ်ထားသင့်ပါသည်။ အရေးကြီးသော သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူ့ဘာသာပြန်အကူအညီကို လိုအပ်ပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းအတွက် ဖြစ်ပေါ်နိုင်သည့် သဘောထားမှားခြင်း သို့မဟုတ် ဘာသာပြန်မှားခြင်းများအတွက် ကျွန်ုပ်တို့သည် တာဝန်မရှိပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
